# Colab AI Chat Connector

Tap ▶ on the next cell. The Vercel app copies a short-lived pairing token before opening this notebook.

The connector tries to read that token from your clipboard automatically. If iPhone/Colab blocks clipboard access, paste the token once when asked.

**No email, password, Supabase user token, or private key is stored in this notebook.**

In [ ]:
import json, time, urllib.request, urllib.error
from datetime import datetime, timezone

try:
    from google.colab import ai, output
except Exception as exc:
    raise RuntimeError("This connector must run inside Google Colab.") from exc

SUPABASE_URL = "https://udyqxlnexnxhmslsebre.supabase.co"
SUPABASE_KEY = "sb_publishable_IgyY0SZrN1tZUPNRnHWVrw_OOZwEsrH"

def rpc(name, payload):
    body = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(
        SUPABASE_URL + "/rest/v1/rpc/" + name,
        data=body,
        headers={
            "apikey": SUPABASE_KEY,
            "Content-Type": "application/json",
        },
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            raw = resp.read().decode("utf-8")
            return json.loads(raw) if raw else None
    except urllib.error.HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Connector HTTP {exc.code}: {detail}") from None

def get_pair_token():
    try:
        value = output.eval_js("navigator.clipboard.readText()")
        if isinstance(value, str):
            value = value.strip()
            if value.startswith("AIWB-") and len(value) >= 29:
                print("Pairing token read from clipboard ✓")
                return value
    except Exception:
        pass
    print("Clipboard access was blocked by the browser.")
    return input("Paste the pairing token copied by the app: ").strip()

pair_token = get_pair_token()
started_at = datetime.now(timezone.utc).isoformat()

def heartbeat(status, include_started=False):
    payload = {"p_token": pair_token, "p_status": status}
    if include_started:
        payload["p_started_at"] = started_at
    ok = rpc("aiwb_pair_heartbeat", payload)
    if ok is not True:
        raise RuntimeError("Pairing token is invalid or expired. Return to the Vercel app and tap Enable Colab again.")

def claim_job():
    jobs = rpc("aiwb_pair_claim_job", {"p_token": pair_token})
    return jobs or []

def finish_job(job_id, result=None, error=None):
    return rpc("aiwb_pair_finish_job", {
        "p_token": pair_token,
        "p_job_id": job_id,
        "p_result": result,
        "p_error": error,
    })

heartbeat("starting", include_started=True)
print("Testing Google Colab AI...")
try:
    test = ai.generate_text("Reply with exactly: COLAB_AI_OK")
    test_text = getattr(test, "text", None) or str(test)
    if "COLAB_AI_OK" not in test_text:
        print("AI responded, but with unexpected text:", test_text[:120])
except Exception as exc:
    try:
        heartbeat("error")
    except Exception:
        pass
    raise RuntimeError("Colab AI test failed: " + str(exc)) from exc

heartbeat("ready")
print("\nCONNECTED ✓")
print("Return to https://colab-ai-workbench.vercel.app")
print("No email or password is stored in Colab.\n")

last_job = time.monotonic()
try:
    while True:
        heartbeat("ready")
        jobs = claim_job()
        if jobs:
            job = jobs[0]
            try:
                heartbeat("busy")
                response = ai.generate_text(job["prompt"])
                answer = getattr(response, "text", None) or str(response)
                finish_job(job["id"], result=answer[:100000])
            except Exception as exc:
                finish_job(job["id"], error=str(exc)[:2000])
            finally:
                last_job = time.monotonic()
                heartbeat("ready")
        else:
            if time.monotonic() - last_job > 1800:
                print("Idle 30 minutes — connector stopped. Tap Enable Colab again when needed.")
                break
            time.sleep(1)
finally:
    try:
        heartbeat("stopped")
    except Exception:
        pass
